# Week 3 - Data Preprocessing (v2)

Create Dataset A with only the rules marked **Discussed**. Each short code cell prints a result that can be checked before continuing. Pending fields and experiments are saved for later decisions.

In [ ]:
# Import packages and locate the raw and processed data folders.
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.paths import sold_data_dir

DATA_DIR = sold_data_dir()
OUTPUT_DIR = REPO_ROOT / "data" / "processed"
print("Raw data folder found:", DATA_DIR.exists())
print("Processed data folder: data/processed")

In [ ]:
# Sort the monthly files and reserve the two newest months for validation and testing.
def source_month(file):
    stamp = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    return pd.Period(f"{stamp[:4]}-{stamp[4:6]}", freq="M")

csv_files = sorted(DATA_DIR.glob("CRMLSSold*.csv"), key=source_month)
assert len(csv_files) >= 3, "At least three monthly files are required."

TRAIN_WINDOW_MONTHS = None  # Later try 6, 12, 18, or 24.
available_train_files = csv_files[:-2]
train_files = (
    available_train_files
    if TRAIN_WINDOW_MONTHS is None
    else available_train_files[-TRAIN_WINDOW_MONTHS:]
)
validation_file = csv_files[-2]
test_file = csv_files[-1]

split_plan = pd.DataFrame({
    "Split": ["Training", "Validation", "Test"],
    "Months": [
        f"{source_month(train_files[0])} to {source_month(train_files[-1])}",
        str(source_month(validation_file)),
        str(source_month(test_file)),
    ],
    "Files": [len(train_files), 1, 1],
})
print(f"Found {len(csv_files)} monthly files.")
display(split_plan)

In [ ]:
# Separate the discussed columns from fields that still need a team decision.
TARGET = "ClosePrice"
FILTER_COLUMNS = ["StateOrProvince", "PropertyType", "PropertySubType"]
DATE_COLUMNS = ["CloseDate", "ListingContractDate"]
BINARY_FEATURES = [
    "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "PoolPrivateYN", "ViewYN",
]
MODEL_FEATURES = ["LivingArea"] + BINARY_FEATURES
LOAD_COLUMNS = ["ListingKey"] + FILTER_COLUMNS + DATE_COLUMNS + MODEL_FEATURES + [TARGET]

PENDING_FEATURES = [
    "BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces",
    "ParkingTotal", "Stories", "YearBuilt / PropertyAgeAtSale",
    "LotSizeSquareFeet", "LotSizeAcres", "Latitude", "Longitude",
    "City", "CountyOrParish", "PostalCode", "Flooring",
]

column_plan = pd.DataFrame({
    "Group": ["Filters", "Dates", "Target", "Model features", "Pending"],
    "Columns": [
        ", ".join(FILTER_COLUMNS),
        ", ".join(DATE_COLUMNS),
        TARGET,
        ", ".join(MODEL_FEATURES),
        ", ".join(PENDING_FEATURES),
    ],
    "Use now?": ["Yes", "Yes", "Yes", "Yes", "No"],
})
display(column_plan)

In [ ]:
# Check the schema, load each split, and report only safe summary information.
for file in csv_files:
    columns = set(pd.read_csv(file, nrows=0).columns)
    missing = sorted(set(LOAD_COLUMNS) - columns)
    assert not missing, f"{file.name} is missing: {missing}"
print("All required columns were found in every monthly file.")

def load_files(files):
    frames = []
    for file in files:
        month = pd.read_csv(file, usecols=LOAD_COLUMNS, low_memory=False)[LOAD_COLUMNS]
        month["SourceMonth"] = str(source_month(file))
        frames.append(month)
    return pd.concat(frames, ignore_index=True)

splits = {
    "Training": load_files(train_files),
    "Validation": load_files([validation_file]),
    "Test": load_files([test_file]),
}
display(pd.Series({name: len(data) for name, data in splits.items()}, name="Raw rows").to_frame())
print("Selected columns:", splits["Training"].columns.tolist())

In [ ]:
# Check missing values before any cleaning is applied.
check_columns = [TARGET] + DATE_COLUMNS + MODEL_FEATURES
raw_missing = pd.DataFrame({
    "Missing rows": splits["Training"][check_columns].isna().sum(),
    "Missing %": splits["Training"][check_columns].isna().mean().mul(100).round(2),
})
display(raw_missing)

In [ ]:
# Convert prices, living area, dates, and binary fields to consistent types.
BINARY_MAP = {
    "true": 1.0, "yes": 1.0, "y": 1.0, "1": 1.0, "1.0": 1.0,
    "false": 0.0, "no": 0.0, "n": 0.0, "0": 0.0, "0.0": 0.0,
}

def convert_types(data):
    converted = data.copy()
    converted[TARGET] = pd.to_numeric(converted[TARGET], errors="coerce")
    converted["LivingArea"] = pd.to_numeric(converted["LivingArea"], errors="coerce")
    converted[DATE_COLUMNS] = converted[DATE_COLUMNS].apply(pd.to_datetime, errors="coerce")
    for column in BINARY_FEATURES:
        values = converted[column].astype("string").str.strip().str.lower()
        unexpected = sorted(values[values.notna() & ~values.isin(BINARY_MAP)].unique())
        assert not unexpected, f"Unexpected {column} values: {unexpected}"
        converted[column] = values.map(BINARY_MAP).astype("float64")
    return converted

splits = {name: convert_types(data) for name, data in splits.items()}
display(splits["Training"][[TARGET] + DATE_COLUMNS + MODEL_FEATURES].dtypes.to_frame("Training data type"))

In [ ]:
# Define a small report used after each cleaning rule.
def show_change(step, before):
    result = pd.DataFrame({
        "Before": before,
        "After": {name: len(data) for name, data in splits.items()},
    })
    result["Removed"] = result["Before"] - result["After"]
    print(step)
    display(result)

In [ ]:
# Keep only California Residential / SingleFamilyResidence records.
before = {name: len(data) for name, data in splits.items()}
for name, data in splits.items():
    keep = (
        data["StateOrProvince"].astype("string").str.strip().str.upper().eq("CA")
        & data["PropertyType"].astype("string").str.strip().eq("Residential")
        & data["PropertySubType"].astype("string").str.strip().eq("SingleFamilyResidence")
    )
    splits[name] = data.loc[keep].copy()
show_change("After the California single-family filter", before)

In [ ]:
# Remove rows with a missing, zero, or negative ClosePrice.
before = {name: len(data) for name, data in splits.items()}
for name, data in splits.items():
    keep = data[TARGET].notna() & data[TARGET].gt(0)
    splits[name] = data.loc[keep].copy()
show_change("After the ClosePrice validity rule", before)

In [ ]:
# Remove rows where CloseDate is earlier than ListingContractDate.
before = {name: len(data) for name, data in splits.items()}
for name, data in splits.items():
    both_dates = data[DATE_COLUMNS].notna().all(axis=1)
    invalid_date = both_dates & data["CloseDate"].lt(data["ListingContractDate"])
    splits[name] = data.loc[~invalid_date].copy()
show_change("After the date-consistency rule", before)

In [ ]:
# Remove reported LivingArea values that are zero or negative.
before = {name: len(data) for name, data in splits.items()}
for name, data in splits.items():
    invalid_area = data["LivingArea"].notna() & data["LivingArea"].le(0)
    splits[name] = data.loc[~invalid_area].copy()
show_change("After the LivingArea validity rule", before)

In [ ]:
# Learn ClosePrice limits from training and apply the same limits to every split.
price_lower, price_upper = splits["Training"][TARGET].quantile([0.005, 0.995]).tolist()
print(f"Training 0.5th percentile:  ${price_lower:,.0f}")
print(f"Training 99.5th percentile: ${price_upper:,.0f}")

before = {name: len(data) for name, data in splits.items()}
for name, data in splits.items():
    keep = data[TARGET].between(price_lower, price_upper, inclusive="both")
    splits[name] = data.loc[keep].copy()
show_change("After applying the frozen ClosePrice limits", before)

In [ ]:
# Inspect feature missingness in training only; notebook 03 handles it inside its pipeline.
training_missing = pd.DataFrame({
    "Missing rows": splits["Training"][MODEL_FEATURES].isna().sum(),
    "Missing %": splits["Training"][MODEL_FEATURES].isna().mean().mul(100).round(2),
})
display(training_missing)
print("Detailed feature inspection is limited to training data.")
print("Missing values remain for notebook 03 to handle without leakage.")

In [ ]:
# Check the finished datasets and print the final row and month summary.
expected_months = {
    "Training": {str(source_month(file)) for file in train_files},
    "Validation": {str(source_month(validation_file))},
    "Test": {str(source_month(test_file))},
}
for name, data in splits.items():
    assert set(data["SourceMonth"].unique()) == expected_months[name]
    assert data[TARGET].between(price_lower, price_upper).all()
    assert data["LivingArea"].dropna().gt(0).all()
    both_dates = data[DATE_COLUMNS].notna().all(axis=1)
    assert not (both_dates & data["CloseDate"].lt(data["ListingContractDate"])).any()
    for column in BINARY_FEATURES:
        assert set(data[column].dropna().unique()).issubset({0.0, 1.0})

final_summary = pd.DataFrame([
    [name, data["SourceMonth"].min(), data["SourceMonth"].max(), len(data)]
    for name, data in splits.items()
], columns=["Split", "First month", "Last month", "Rows"]).set_index("Split")
display(final_summary)
print("All fixed-cleaning checks passed.")

In [ ]:
# Optionally save Dataset A after reviewing all intermediate results above.
SAVE_OUTPUTS = False
EXPORT_COLUMNS = (
    DATE_COLUMNS + ["ListingKey", "SourceMonth"]
    + FILTER_COLUMNS + MODEL_FEATURES + [TARGET]
)
output_files = {
    "Training": "train_clean.csv",
    "Validation": "validation_clean.csv",
    "Test": "test_clean.csv",
}

if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    for name, filename in output_files.items():
        splits[name][EXPORT_COLUMNS].to_csv(
            OUTPUT_DIR / filename, index=False, date_format="%Y-%m-%d"
        )
    print("Saved train, validation, and test CSV files to:", OUTPUT_DIR)
else:
    print("Dry run complete. Set SAVE_OUTPUTS = True after reviewing the results.")